<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-6-evaluation-monitoring/Module_6_Session_1_LangSmith_Tracing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6 — Session 1: LangSmith Tracing

## What we're building
So far we've built RAG pipelines and agents — but when something breaks inside,
we can't *see* what happened. LangSmith is a flight recorder for LLM apps:
it auto-logs every prompt, response, token count, and timing to a dashboard.

## The key idea
We don't change our code. We just set a few environment variables, and
LangSmith silently records every LLM call. Like a CCTV camera in the kitchen —
we cook the same way, but now everything is on record.

## AWS equivalent
AWS X-Ray (distributed tracing) + CloudWatch (logs & metrics)

## Step 0: Install libraries
Colab resets every session, so we reinstall our tools each time.
- langchain-groq → lets LangChain talk to Groq (we've used this before)
- langsmith → the tracing library, NEW this session (the flight recorder)

In [ ]:
# The -q flag means "quiet" — less install spam in the output
!pip install -q langchain-groq langsmith
print("✅ Libraries installed")

## Step 1: Load keys and turn on LangSmith tracing
We load our two API keys, then set four environment variables.
These four variables are LangSmith's "ON switch" — once set,
every LLM call gets recorded automatically.

In [ ]:
import os
from google.colab import userdata

# --- Load our two keys from Colab Secrets ---
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

# --- The four "ON switch" variables for LangSmith ---
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"  # ← APAC region (was US)
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "swiggy-module-6"

print("✅ Keys loaded, tracing ON, pointing at APAC region")
print("Endpoint:", os.environ["LANGSMITH_ENDPOINT"])

## Step 2: Make an LLM call and watch it get traced
We make a normal Swiggy support call to Groq — exactly the way we always have.
We change NOTHING about our code. Because tracing is ON, LangSmith records
it silently in the background.

In [ ]:
from langchain_groq import ChatGroq          # our familiar Groq wrapper for LangChain

# Set up the model exactly as in earlier modules
llm = ChatGroq(
    model="openai/gpt-oss-20b",              # our text model (llama was deprecated)
    temperature=0.3                          # low temp = consistent support replies
)

# A normal Swiggy customer complaint
complaint = "My Swiggy order is 40 minutes late and the food is cold. I want a refund."

# Make the call — .invoke() sends the prompt and waits for the reply
response = llm.invoke(complaint)

# Print just the text content of the reply
print(response.content)

In [ ]:
# Check the key WITHOUT printing the whole thing (never print full keys)
key = os.environ["LANGSMITH_API_KEY"]
print("Key length:", len(key))            # a valid lsv2 key is usually ~50+ chars
print("Starts with:", key[:8])            # should show: lsv2_pt_ or lsv2_sk_
print("Ends with:", key[-4:])             # just to confirm nothing got clipped

In [ ]:
from langchain_groq import ChatGroq
from langsmith import Client

# Make a fresh call so a trace is generated under the NEW endpoint
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)
response = llm.invoke("Test trace: my Zomato order is late, I want a refund.")
print("LLM replied:", response.content[:80], "...")

# Force LangSmith to finish sending any pending traces (flush the background queue)
client = Client()
client.flush()
print("✅ Traces flushed to LangSmith")

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langsmith import Client

# Load keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

# LangSmith settings — no WORKSPACE_ID needed for workspace-scoped Personal token
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "swiggy-module-6"

print("Key starts with:", userdata.get("LANGSMITH_API_KEY")[:8])

# Test if the key can see projects (no 403 = key is working)
client = Client(
    api_url="https://apac.api.smith.langchain.com",
    api_key=os.environ["LANGSMITH_API_KEY"],
)

print("Testing key access...")
try:
    projects = list(client.list_projects())
    print("✅ Key works! Projects found:", len(projects))
    for p in projects:
        print(" -", p.name)
except Exception as e:
    print("❌ Still failing:", str(e)[:150])

In [ ]:
!pip install -q langchain-groq langsmith
print("✅ Installed")

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langsmith import Client

# Step 1: Set ALL env vars BEFORE importing anything LangChain-related
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "swiggy-module-6"

print("✅ Env vars set")
print("Endpoint:", os.environ["LANGSMITH_ENDPOINT"])

# Step 2: Make the LLM call — tracer picks up env vars set above
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)
response = llm.invoke("My Swiggy order arrived cold and 45 mins late. I want a refund.")
print("LLM replied:", response.content[:80], "...")

# Step 3: Flush and verify
client = Client(
    api_url="https://apac.api.smith.langchain.com",
    api_key=os.environ["LANGSMITH_API_KEY"],
)
client.flush()
print("✅ Flushed")

# Step 4: Check if project now exists
projects = list(client.list_projects())
print("Projects now visible:", [p.name for p in projects])

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langsmith import Client, traceable

# Set env vars
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

# Build client explicitly pointing at APAC — this is our single source of truth
client = Client(
    api_url="https://apac.api.smith.langchain.com",
    api_key=os.environ["LANGSMITH_API_KEY"],
)

# Set up LLM
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

# @traceable sends this function's trace directly through our client
# no background tracer involved — we control exactly where it goes
@traceable(client=client, project_name="swiggy-module-6")
def handle_complaint(complaint: str) -> str:
    """Swiggy support handler — traces directly to APAC LangSmith"""
    response = llm.invoke(complaint)
    return response.content

# Run it
result = handle_complaint("My Swiggy order arrived cold and 45 mins late. I want a refund.")
print("Reply:", result[:80], "...")

# Flush and verify
client.flush()
projects = list(client.list_projects())
print("Projects now visible:", [p.name for p in projects])

## Step 3: Tracing a Multi-Step Chain
A single LLM call trace is useful. But the real power is when you have
multiple steps — extract complaint details, then generate a reply.
LangSmith shows each step nested inside the parent, with its own latency
and tokens. This is how you debug RAG pipelines and agents in production.

In [ ]:
# A two-step Swiggy support chain — both steps traced and nested

@traceable(client=client, project_name="swiggy-module-6", name="step1_extract_details")
def extract_details(complaint: str) -> str:
    """Step 1: Extract key details from the complaint"""
    prompt = f"""Extract these details from the complaint in JSON format:
    - issue_type (late_delivery / wrong_order / cold_food / missing_items)
    - severity (high / medium / low)
    - wants_refund (true / false)

    Complaint: {complaint}

    Reply with JSON only."""

    response = llm.invoke(prompt)
    return response.content

@traceable(client=client, project_name="swiggy-module-6", name="step2_generate_reply")
def generate_reply(complaint: str, details: str) -> str:
    """Step 2: Generate empathetic reply using extracted details"""
    prompt = f"""You are a Swiggy support agent.

    Customer complaint: {complaint}
    Extracted details: {details}

    Write a short, empathetic reply addressing their specific issue."""

    response = llm.invoke(prompt)
    return response.content

@traceable(client=client, project_name="swiggy-module-6", name="full_support_pipeline")
def full_support_pipeline(complaint: str) -> str:
    """Full pipeline: extract details → generate reply"""
    # Step 1
    details = extract_details(complaint)
    print("Details extracted:", details[:100])

    # Step 2
    reply = generate_reply(complaint, details)
    print("Reply generated:", reply[:100])

    return reply

# Run the full pipeline
complaint = "My Zepto order had 3 missing items and I was charged the full amount!"
result = full_support_pipeline(complaint)

# Flush to LangSmith
client.flush()
print("\n✅ Full pipeline traced — check LangSmith for nested spans")

## Session Summary

### What we built
A LangSmith tracing setup for a Swiggy support pipeline:
1. Single LLM call traced with @traceable decorator
2. Multi-step pipeline traced with nested spans:
   - step1_extract_details → extracts issue_type, severity, wants_refund
   - step2_generate_reply → generates empathetic customer reply
3. Both steps visible on LangSmith dashboard with latency and token counts

### Key concepts
- @traceable decorator: traces any Python function directly to LangSmith
- Nested spans: parent function contains child LLM calls — each with own metrics
- P50/P99 latency: production monitoring metrics visible on dashboard
- APAC region: LangSmith endpoint must match your account's region

### Why @traceable over auto-logging
Auto-logging relies on a background tracer that can cache the wrong endpoint.
@traceable with explicit client=client gives precise, production-correct control
over exactly which functions get traced and exactly where traces go.

### AWS equivalent
- AWS X-Ray → distributed tracing (same concept as LangSmith nested spans)
- Amazon CloudWatch → metrics dashboard (P50/P99 latency, error rates)
- Together they give the same observability LangSmith provides for LLM apps

### What is next
Module 6 Session 2 — Opik Monitoring:
A second observability tool focused on LLM quality metrics,
not just latency and tokens.